In [ ]:
import datetime
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

def get_time(city: str) -> str:
    """获取时间"""
    return f"{city} 现在时间是：{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

client = OpenAI()

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_time",
            "description": "获取指定城市的当前时间",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，例如北京、上海"
                    }
                },
                "required": ["city"],
                "additionalProperties": False
            }
        }
    }
]

messages = [
    {"role": "user", "content": "北京，现在几点了？"}
]

response = client.chat.completions.create(
    model="qwen3.5-flash",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
messages.append(message.model_dump(exclude_none=True))

function_map = {
    "get_time": get_time,
}

if message.tool_calls:
    print("=== 工具调用 ===")
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        print(f"函数: {tool_call.function.name}")
        print(f"参数: {tool_call.function.arguments}")

        result = function_map[tool_call.function.name](**arguments)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            }
        )

    final_response = client.chat.completions.create(
        model="qwen3.5-flash",
        messages=messages,
    )
    print("=== 响应内容 ===")
    print(final_response.choices[0].message.content)
else:
    print("=== 响应内容 ===")
    print(message.content or "无")
    

=== 工具调用 ===
函数: get_time
参数: {"city": "北京"}
=== 响应内容 ===
北京现在的时间是：**2026 年 4 月 11 日 14:30（下午 2:30）**。

希望您的一天愉快！
[{'role': 'user', 'content': '北京，现在几点了？'}, {'content': '', 'role': 'assistant', 'tool_calls': [{'id': 'call_bdf55253d8e8498294cc3f0f', 'function': {'arguments': '{"city": "北京"}', 'name': 'get_time'}, 'type': 'function', 'index': 0}], 'reasoning_content': '用户询问北京现在的时间。我需要使用get_time工具来获取北京的当前时间，参数city应该是"北京"。'}, {'role': 'tool', 'tool_call_id': 'call_bdf55253d8e8498294cc3f0f', 'content': '北京 现在时间是：2026-04-11 14:30:02'}]
